In [ ]:
import numpy as np
from pathlib import Path
import prism

from imagematerials.factory import ModelFactory
from imagematerials.model import RestOf
from imagematerials.preprocessing import get_preprocessing_data
from imagematerials.rest_of.preprocessing.main import fit_all_materials_save_corrseponding_input_data
from imagematerials.rest_of.util import sum_inflows_for_all_sectors

from imagematerials.rest_of.preprocessing.resource_efficiency_measures import calcualte_resource_efficiency_measures, read_gompertz_coefs

In [ ]:
# REFIT DATA
refit = False

if refit == True:
    fit_all_materials_save_corrseponding_input_data(path_input_data=Path("../data/raw/rest-of"),
                                                    path_input_data_image=Path("../data/raw/image"))
    
# REFIT EFFICIENCY MEASURES
refit_efficiency_measures = False
if refit_efficiency_measures == True:
    calcualte_resource_efficiency_measures()

In [6]:
# Define the complete timeline, including historic tail

YEAR_START  = 1971    # start year of the simulation period
YEAR_END    = 2100    # end year of the calculations
YEAR_OUT    = 2100    # year of output generation = last year of reporting
time_start = 1971
complete_timeline = prism.Timeline(time_start, YEAR_END, 1)
simulation_timeline = prism.Timeline(YEAR_START, YEAR_OUT, 1)

In [ ]:
scenario_list = {"SSP2_baseline":("SSP2_baseline", None)}

In [8]:

# Define the scenario list
scenario_list = {       
                "base":("SSP2",["base"])
                }

# Define paths
scenario_base_path = Path("..", "data", "raw")
CP_scenario_path = scenario_base_path/ "IMAGE_CircoMod"
CE_scenario_path = scenario_base_path / 'circular_economy_scenarios'


In [9]:
# Run simulations for all scenarios

model_runs = {} # to store outputs for all scenarios

# Loop over scenarios
for scen_id, (climate_scen, circular_scen) in scenario_list.items():
    print(f"Running scenario: {climate_scen} from {CP_scenario_path}")
    print(f"Circular economy scenario: {circular_scen}")
    climate_policy_scenario_dir = CP_scenario_path / climate_scen           # path to climate policy scenario data (i.e., from IMAGE)
    circular_economy_scenario_dirs = {
        scenario: CE_scenario_path / scenario for scenario in circular_scen # path to circular economy scenario data
    }

# Get preprocessing data for all sectors and adjust material indices
# #REST OF
    rest_sector = get_preprocessing_data(sector="rest_of",base_dir=scenario_base_path,
                                         climate_policy_scenario_dir=climate_policy_scenario_dir,
                                         circular_economy_scenario_dirs=circular_economy_scenario_dirs,
                                         scenario_name=climate_scen)
    

# Build and run the model
    factory = ModelFactory(rest_sector,complete_timeline
                           ).add(RestOf)                

    model = factory.finish() # create the model
    model_runs[climate_scen] = model
    
    # Run the simulation (suppressing warnings)
    import warnings
    with warnings.catch_warnings():
        warnings.filterwarnings("ignore")
        model.simulate(simulation_timeline) 
    
    print(f"Finished {scen_id}")


Running scenario: SSP2 from ..\data\raw\IMAGE_CircoMod
Circular economy scenario: ['base']
Finished base


In [10]:
model_runs.get('SSP2').rest_of.get('inflow_materials')

Magnitude,[[[224679.8674889295 7024224.554116073 2591000.0 ... 6191035.574 246807734.52720854 8558308.67611027] [2082780.6925649357 32155284.163083706 66821083.415 ... 565559383.043 1571615051.1227415 63180028.09324152] [532569.5914389679 6915732.973661226 2132826.661 ... 8787660.161 12465149.944314077 3938100.9437340754] ... [260225.39475213172 2745168.190182759 10825467.189999998 ... 13637130.918864 95832503.07148838 3749779.625857353] [287309.35180912423 137308.97497447213 965952.427 ... 3947292.157 7434674.496277783 4.124141175759748] [nan 8092361.122748168 1942584.769 ... 3384022.3379 834583.4311126359 4.174845956865246]] [[228103.54234599828 4925874.592849596 5074449.248 ... 7171688.818 277113264.4658494 10174219.822323896] [2106702.483705381 28730752.897057764 72040792.283 ... 600821696.299 1714247372.7541149 106321327.49768062] [549519.751261582 1944368.383263439 2450651.228 ... 10163675.016 41865279.603860736 696593.3872532607] ... [266134.19463590597 4345708.474069845 12472892.126 ... 14139411.406514 113605634.20462215 7713242.963826444] [293218.68702570256 16641.87683931412 981472.049 ... 4083109.006800001 17599547.95757375 191070.28832514584] [nan 569030.805088904 2215701.357 ... 4339108.32 14767394.69051927 1996566.7126591601]] [[230975.69698919542 7059423.607047452 5798794.256 ... 8713331.333 295183973.27550375 12085505.63456472] [2128016.024628236 31124263.518427588 75489450.644 ... 692723053.604 1813352480.510724 116996195.46659404] [566848.6621429806 2902741.625883394 2766046.917 ... 11797833.86 48768438.428804696 1085063.186817423] ... [270757.34455035825 3891389.110677762 9973714.689 ... 15879313.56386 120980996.22895458 6566690.080308289] [300859.0366344148 1201715.4779716576 1051179.127 ... 4454927.523399999 19447480.587770343 41130.246056998614] [nan 918328.0889800251 2230094.161 ... 4490967.3952 15489571.644040633 2560047.3240431743]] ... [[457864.2114921684 14314326.74578618 9321390.131454779 ... 15390760.101520687 557573499.9999956 17440562.427903213] [4173207.530693353 64434096.539065234 84959895.35049841 ... 1207601952.904131 2536011472.107272 127655379.82689683] [1508061.3284877264 23284384.61571628 48717680.65864053 ... 113733917.87639576 1482966214.2642498 16094716.023714183] ... [687260.6861843476 7261403.982448467 19964440.40070453 ... 35483659.83010888 285797195.82573813 9903246.072841598] [1439585.540657404 131926208.22058947 276027817.40948844 ... 287216039.56974965 3429609913.8537917 45466102.77975735] [nan 8092361.122748168 172251259.605127 ... 391453079.6048472 1819141985.7574143 26204058.40451252]] [[457232.5242891533 14294578.146096282 9308529.980535446 ... 15369526.412674114 556804249.9999956 17416500.752361488] [4171393.7651949334 64406092.09874807 84922969.95778939 ... 1207077103.19515 2534911713.2445273 127599898.06091984] [1504614.4555144978 23231165.085152928 48606330.610246286 ... 113478134.315589 1488694249.0432732 16057929.42875056] ... [686846.836192667 7257031.359311227 19952418.349014636 ... 35462292.95810661 285625358.8240616 9897282.60310261] [1439075.1932027186 131879439.06483543 275929982.2435894 ... 287114218.6146209 3433967880.262284 45460736.7908193] [nan 8092361.122748168 172484831.1476814 ... 393330651.47524095 1851921802.535141 26551367.94585037]] [[456600.8370861382 14274829.546406383 9295669.829616113 ... 15348292.72382754 556034999.9999957 17392439.07681976] [4169579.999696513 64378087.65843091 84886044.5650799 ... 1206552253.4861686 2533811773.4746103 127544416.29494156] [1501168.6090073127 23177961.403157983 48495013.62640638 ... 113222062.16046496 1494214103.2005346 16021153.788635269] ... [686433.1362009834 7252660.321031955 19940400.65471934 ... 35440933.76017495 285453556.44433546 9891321.294824202] [1438564.845748033 131832669.90775552 275832141.48340535 ... 287012397.65949214 3437601006.7906055 45453207.19085905] [nan 8092361.122748168 172717246.3747372 ... 395041277.5052504 1882275839.8014174 26855925.826332636]]]
Units,metric_ton
